# Clase 005 — VS Code / Cursor para Python y Jupyter

**Parte 0 — Prerrequisitos** · VS Code Python docs.

> 🎯 Configurar VS Code como IDE serio para Python+Jupyter: intérprete por workspace, debugger gráfico, ruff, tests integrados.

> ⏱️ ~60 min

## ⚙️ Setup mínimo

**Extensiones obligatorias:**
- `ms-python.python` — soporte Python
- `ms-toolsai.jupyter` — notebooks nativos
- `charliermarsh.ruff` — linter + formatter
- `tamasfe.even-better-toml` — soporte `pyproject.toml`
- `eamodio.gitlens` — git superpoderes

Instala todas con:

```bash
code --install-extension ms-python.python
code --install-extension ms-toolsai.jupyter
code --install-extension charliermarsh.ruff
code --install-extension tamasfe.even-better-toml
code --install-extension eamodio.gitlens
```

## 1️⃣ Selector de intérprete (el bug invisible)

VS Code recuerda **un intérprete por workspace** en `.vscode/settings.json`. Si no lo configuras, usará el primero que encuentre — generalmente el del sistema. Resultado: `import` funciona en terminal pero no en VS Code (o al revés).

**Cómo configurarlo bien:**

```json
// .vscode/settings.json
{
  "python.defaultInterpreterPath": "${workspaceFolder}/.venv/bin/python",
  "python.terminal.activateEnvironment": true,
  "editor.formatOnSave": true,
  "[python]": {
    "editor.defaultFormatter": "charliermarsh.ruff",
    "editor.codeActionsOnSave": {
      "source.fixAll.ruff": "explicit",
      "source.organizeImports.ruff": "explicit"
    }
  }
}
```

En Windows: `"${workspaceFolder}/.venv/Scripts/python.exe"`.

In [ ]:
import sys
from pathlib import Path

print('Intérprete que ejecuta esta celda:')
print(f'  {sys.executable}')
print()

in_venv = sys.prefix != sys.base_prefix
print(f'¿Estás en un venv? {in_venv}')
if in_venv:
    print(f'  prefix: {Path(sys.prefix).name}')
else:
    print('⚠️  Selecciona un intérprete del venv del proyecto en VS Code.')

## 2️⃣ Debugger gráfico — el fin de los `print("AQUI 1")`

El debug por `print` es lento y no escala. VS Code te da:
- **Breakpoints** (F9): para la ejecución en la línea
- **Step over** (F10): siguiente línea
- **Step into** (F11): entra a la función
- **Step out** (Shift+F11): sale de la función
- **Variables** (panel izquierdo): valores actuales en el scope
- **Watch**: expresiones que evalúas en tiempo real
- **Call stack**: cómo llegaste aquí

Config mínima (`.vscode/launch.json`):

```json
{
  "version": "0.2.0",
  "configurations": [
    {
      "name": "Python: Archivo actual",
      "type": "debugpy",
      "request": "launch",
      "program": "${file}",
      "console": "integratedTerminal",
      "justMyCode": false
    }
  ]
}
```

`justMyCode: false` te deja entrar a código de librerías (útil cuando un error viene de pandas).

## 3️⃣ Notebooks nativos en VS Code

Mejor UX que Jupyter web para edición:
- Autocompletado con type hints reales (no solo nombres de variables)
- Hover muestra docstring de funciones de pandas/sklearn
- Debug de celda con breakpoint
- Git integrado (ves diffs por celda)
- Outline lateral con headers de markdown

Selector de kernel arriba a la derecha: elige el mismo intérprete del workspace para que `pip install` funcione consistente.

In [ ]:
# Demo: autocompletado funciona con type hints
from typing import Iterable

def promedio(xs: Iterable[float]) -> float:
    """Promedio aritmético — escribe `promedio(` y mira el hint."""
    xs = list(xs)
    return sum(xs) / len(xs) if xs else 0.0

print(promedio([1, 2, 3, 4, 5]))
print(promedio.__doc__)

## 4️⃣ ruff — un solo tool reemplaza 4

En 2026, **ruff** (Astral, Rust) sustituye al stack tradicional:
- ❌ `black` (formatter) → ✅ `ruff format`
- ❌ `isort` (import sort) → ✅ `ruff check --select I --fix`
- ❌ `flake8` (linter) → ✅ `ruff check`
- ❌ `pylint` (linter más estricto) → ✅ `ruff check --select PL`

Ventaja: 10–100× más rápido, 1 tool, 1 config.

Config recomendada (`pyproject.toml`):

```toml
[tool.ruff]
line-length = 100
target-version = "py312"

[tool.ruff.lint]
select = [
    "E",   # pycodestyle errors
    "F",   # pyflakes
    "I",   # isort
    "UP",  # pyupgrade (sintaxis moderna)
    "B",   # flake8-bugbear (bugs comunes)
    "N",   # pep8-naming
]
ignore = ["E501"]  # line-too-long lo deja al formatter

[tool.ruff.format]
quote-style = "double"
```

## 5️⃣ Tests integrados

Panel **Testing** (icono matraz). Con `pytest` instalado y tests en `tests/`:
- VS Code descubre automáticamente
- Click derecho → "Run Test" o "Debug Test"
- Output inline (verde/rojo) en el archivo
- Coverage opcional con `coverage.py` extension

Config (`pyproject.toml`):

```toml
[tool.pytest.ini_options]
testpaths = ["tests"]
python_files = "test_*.py"
addopts = "-v --tb=short"
```

## 6️⃣ ¿Cuándo Cursor en vez de VS Code?

**Cursor** = fork de VS Code con IA integrada (chat con contexto del proyecto, edición multi-archivo, autocompletado avanzado).

**Usa Cursor si:**
- Quieres pair programming con IA sin saltar a otra app
- Trabajas mucho en refactors o exploración de código nuevo
- Estás OK con pagar la suscripción

**Quédate con VS Code si:**
- Tu organización tiene políticas estrictas sobre IA
- Ya pagas Copilot y te alcanza
- No quieres dependencias adicionales

Ambos comparten extensiones — migrar es trivial.

## ✅ Checklist

- [ ] Mi VS Code apunta al intérprete del venv del proyecto
- [ ] Sé poner un breakpoint y debuggear sin `print`
- [ ] Edito notebooks en VS Code con autocompletado
- [ ] Tengo ruff configurado en `pyproject.toml`
- [ ] Sé correr tests desde el panel Testing

## 📝 Homework

Ver `README.md`. Repo con `.vscode/settings.json`, `pyproject.toml` con ruff, y screenshot del debugger en acción.

## 📖 Definiciones y características

**Workspace**

Concepto de VS Code = una carpeta (o conjunto de carpetas) con configuración asociada en `.vscode/settings.json`. La configuración del workspace **override** a la del usuario. Característica: pones `.vscode/` en git para que todos los colaboradores hereden la misma config.

**Intérprete Python**

Ejecutable concreto (`/path/to/.venv/bin/python`). VS Code recuerda **uno por workspace**. Es el origen del 90% de los "funciona en mi máquina" entre IDE y terminal.

**ruff**

Linter + formatter en un solo binario, escrito en Rust. Reemplaza black + isort + flake8 + (parte de) pylint con un único tool 10–100× más rápido. Config en `[tool.ruff]` de `pyproject.toml`.

**Breakpoint**

Marca en una línea (F9) que pausa la ejecución cuando llega ahí. Permite inspeccionar variables, paso a paso, evaluar expresiones — mil veces más eficiente que `print`.

**`launch.json`**

Config de debug de VS Code. Define perfiles: "debug archivo actual", "debug tests", "debug Django", etc. Cada perfil tiene su `program`, `args`, `env`, `justMyCode`.

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| "Python interpreter is not selected" al abrir un .py | Workspace nuevo, VS Code no eligió uno. **Fix**: `Ctrl+Shift+P` → "Python: Select Interpreter" → elige el del `.venv` del proyecto. Guarda en `.vscode/settings.json` para que persista. |
| Format-on-save no aplica ruff aunque está instalado | Falta declarar ruff como formatter por default para Python. **Fix**: en `settings.json`, `"[python]": { "editor.defaultFormatter": "charliermarsh.ruff" }` y `"editor.formatOnSave": true`. |
| Debugger arranca pero se salta mis breakpoints | Estás corriendo el archivo (Ctrl+F5 = sin debug) en vez de debug (F5). O `justMyCode: true` está saltando código que vive en librerías que sí querías inspeccionar. |
| Tests no aparecen en el panel "Testing" | VS Code no detectó pytest. **Fix**: `Ctrl+Shift+P` → "Python: Configure Tests" → pytest → carpeta `tests`. O añade `[tool.pytest.ini_options] testpaths = ["tests"]` en `pyproject.toml`. |
| Cambié interpreter y los imports siguen rotos | VS Code cachea symbols del intérprete viejo. **Fix**: `Ctrl+Shift+P` → "Python: Restart Language Server". Si persiste, recarga la ventana (`Reload Window`). |

## ❓ Preguntas frecuentes

**❓ ¿VS Code o Cursor?**

Cursor = VS Code + IA integrada (chat con contexto del repo, edición multi-archivo). Si pagas Copilot o no te interesa IA, quédate en VS Code. Si quieres pair-programming con IA sin saltar a otra app, Cursor. Las extensiones son las mismas.

**❓ ¿Debo commitear `.vscode/`?**

**Sí** la parte compartida: `settings.json` (interpreter path relativo, formatter, etc.), `extensions.json` (recomendaciones). **No** lo personal: `.vscode/launch.json` con paths absolutos del tester.

**❓ ¿Notebook en VS Code o en JupyterLab?**

VS Code para escribir/refactorizar (autocomplete con type hints, debug por celda, git inline). JupyterLab cuando alguien necesita un navegador y no quiere instalar VS Code (alumno, demo en proyector).

**❓ ¿Para qué `justMyCode: false`?**

Por default, el debugger se salta código de librerías de terceros (numpy, pandas) — útil para no perderte. Pero a veces el bug viene **desde dentro de pandas** (datos malformados); con `false` puedes entrar a ver.

**❓ ¿Ruff reemplaza todo el stack? ¿No necesito black?**

Sí — `ruff format` es drop-in replacement de black (mismo output prácticamente). Mismo con isort (`ruff check --select I --fix`) y flake8 (`ruff check`). Único caso donde aún conviene black: si tu org ya tiene CI con black configurado y no quieres tocar.

## 🔗 Referencias

- [VS Code Python tutorial](https://code.visualstudio.com/docs/python/python-tutorial)
- [ruff docs](https://docs.astral.sh/ruff/)

➡️ **Siguiente:** [006 — Python: tipos, estructuras, control de flujo](../006-python-tipos-estructuras-control-de-flujo/README.md)